**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [22]:
import digitalhub as dh

# dh.refresh_token()
profile = dh.get_current_profile()
print(profile)


__default


In [23]:
dh.refresh_token()

In [24]:
config = dh.get_credentials_and_config()

HTTPError: 401 Client Error:  for url: http://rsde-platform-core.rsde-platform.svc.cluster.local:8080/api/auth

In [1]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

/home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub/stores/client/auth/refresh.py:110: UserWarning: Failed to refresh credentials after retry (checked credentials from file and env). Please check your credentials and make sure they are up to date. (refresh tokens, password, etc.).
  warn(


UnauthorizedError: Unauthorized. Response: .

**SETUP PARAMETERS**

In [ ]:
# Parametri Job   
job_name = "test_encoders_weights_v6"                                
dataset = "Test" 
test_sar = False
test_opt = True                                       
#handler = pretrain_encoders                                         

parametri = {     
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                      
    "n_images2": 4, "n_channels2": 10,  
    "output_dim": 512,                              
    "mamba": False, 
    "dataset": dataset,
    "test_sar": test_sar,                                 
    "test_opt": test_opt,
    "weights_s1": "encoder-s1-weights_train_s1_v3_Standard_200",
    "weights_s2": "encoder-s2-weights_train_s2_v3_Standard_200",
    "n_samples": 10          
}                                                              

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

PARAMETRI: {'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'output_dim': 512, 'mamba': False, 'dataset': 'Test', 'test_sar': True, 'test_opt': False, 'weights_s1': 'encoder-s1-weights_train_s1_v3_Standard_200', 'weights_s2': 'encoder-s2-weights_train_s12_v5_Test_20', 'n_samples': 10}


**BUILD ENVIRONMENT**

In [ ]:
test_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="test_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

2026-08-30 07:29:12,329 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:17,341 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:22,355 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:27,370 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:32,384 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:37,399 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07:29:42,414 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8ca4e05c384491a45cf849ca62b59b to finish...
2026-08-30 07

BUILD: COMPLETED


**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)

2026-08-30 07:30:02,773 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:07,783 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:12,796 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:17,810 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:22,822 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:27,834 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07:30:32,883 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7b7262deb73d4005a1dbe6b4099b69a1 to finish...
2026-08-30 07

Run test_encoders avviato: 7b7262deb73d4005a1dbe6b4099b69a1
COMPLETED
None
